In [1]:
# ============ TIR Binary Classification Model (v1) ============
# Migrated from multi v3 - simplified for TIR presence/absence prediction
# Task: Predict TRUE (has TIR) vs FALSE (no TIR)

import os
import gc
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import pandas as pd
from pathlib import Path
from collections import Counter
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.optim.lr_scheduler import CosineAnnealingLR

from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve, 
                             confusion_matrix, accuracy_score, balanced_accuracy_score, 
                             f1_score, classification_report, precision_recall_curve)

# ============ Configuration ============
# Paths
FASTA_PATH = "../../data/vgp/all_vgp_tes.fa"
LABEL_PATH = "../../data/vgp/features"  # TIR labels: TRUE/FALSE

# Fixed canvas length for random placement
FIXED_LENGTH = 40000

# RC fusion strategy: "early" (after first conv) or "late" (after pooling)
RC_FUSION_MODE = "early"

# Label smoothing factor (0.0 = no smoothing)
LABEL_SMOOTHING = 0.1

# Training parameters
BATCH_SIZE = 32
EPOCHS = 1
LR = 1e-3
PATIENCE = 10

def resolve_device(requested=None):
    """Return the best available accelerator as a torch.device."""
    if requested is not None:
        return torch.device(requested)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

DEVICE = resolve_device()
print(f"Using device: {DEVICE}")

Using device: mps


## Load Data

In [2]:
# ============ Data Loading Functions ============

def read_fasta(path):
    """Read FASTA file and return headers and sequences."""
    headers, sequences = [], []
    h, buf = None, []
    
    with open(path, 'r') as f:
        for line in f:
            if not line: 
                continue
            if line[0] == '>':
                if h is not None:
                    sequences.append(''.join(buf).upper())
                    buf = []
                h = line[1:].strip()
                headers.append(h)
            else:
                buf.append(line.strip())
        if h is not None:
            sequences.append(''.join(buf).upper())
            
    return headers, sequences


def load_tir_labels(label_path):
    """
    Load TIR presence/absence labels.
    
    Expected format: >header\tTRUE or >header\tFALSE
    
    Returns:
        label_dict: header -> 1 (TRUE/has TIR) or 0 (FALSE/no TIR)
    """
    label_path = Path(label_path)
    label_dict = {}
    n_true = 0
    n_false = 0
    
    with label_path.open("r") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split()
            if len(parts) < 2:
                continue
            header = parts[0].lstrip('>')
            tag = parts[1].upper()
            
            if tag == "TRUE":
                label_dict[header] = 1
                n_true += 1
            elif tag == "FALSE":
                label_dict[header] = 0
                n_false += 1
            else:
                print(f"Warning: Unknown label '{tag}' for {header}")
                continue
    
    total = n_true + n_false
    print(f"Loaded {total} TIR labels:")
    print(f"  ✓ TRUE (has TIR):  {n_true} ({100*n_true/total:.1f}%)")
    print(f"  ✗ FALSE (no TIR):  {n_false} ({100*n_false/total:.1f}%)")
    
    return label_dict


def compute_binary_class_weights(labels, mode="inv_sqrt", eps=1e-6):
    """
    Compute class weights for imbalanced binary classification.
    """
    labels = np.asarray(labels, dtype=np.int64)
    counts = np.bincount(labels, minlength=2).astype(np.float64)
    
    if mode == "none":
        w = np.ones(2, dtype=np.float32)
    elif mode == "inv":
        w = 1.0 / (counts + eps)
    elif mode == "inv_sqrt":
        w = 1.0 / np.sqrt(counts + eps)
    elif mode == "balanced":
        # sklearn-style balanced weights
        n_samples = len(labels)
        w = n_samples / (2 * counts + eps)
    else:
        raise ValueError(f"Unknown mode={mode}")
    
    w = w / (w.mean() + 1e-12)  # normalize to mean=1
    return w.astype(np.float32)

## Data Processing

In [3]:
# ============ Encoding ============
# Mapping ACGT to 0-3, N to 4
ENCODE = np.full(256, 4, dtype=np.int64)
for ch, idx in zip(b"ACGTNacgtn", [0, 1, 2, 3, 4, 0, 1, 2, 3, 4]):
    ENCODE[ch] = idx

# Reverse complement: ACGTN -> TGCAN -> indices [3, 2, 1, 0, 4]
REV_COMP = torch.tensor([3, 2, 1, 0, 4], dtype=torch.long)

In [4]:
# ============ Dataset for TIR Binary Classification ============

class TIRDataset(Dataset):
    """
    Dataset for TIR presence/absence binary classification.
    
    Each sequence is placed at a random position within a fixed-length canvas.
    This provides data augmentation and forces model to learn position-invariant features.
    """
    def __init__(self, headers, sequences, labels, fixed_length=FIXED_LENGTH):
        self.headers = list(headers)
        self.sequences = list(sequences)
        self.labels = np.asarray(labels, dtype=np.int64)
        self.fixed_length = fixed_length
        self.seq_lengths = np.array([len(s) for s in sequences], dtype=np.int64)
        
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        seq = self.sequences[idx]
        seq_len = len(seq)
        
        # Encode sequence
        seq_bytes = seq.encode("ascii", "ignore")
        seq_idx = ENCODE[np.frombuffer(seq_bytes, dtype=np.uint8)]
        
        # Random placement within canvas
        max_start = max(0, self.fixed_length - seq_len)
        if max_start > 0:
            start_pos = np.random.randint(0, max_start + 1)
        else:
            start_pos = 0
        
        end_pos = start_pos + seq_len
        
        return (
            self.headers[idx],
            seq_idx,
            int(self.labels[idx]),
            start_pos,
            end_pos,
            seq_len
        )


def collate_tir(batch, fixed_length=FIXED_LENGTH):
    """
    Collate function for TIR binary classification.
    
    Returns:
        headers: list of header strings
        X: (B, 5, fixed_length) one-hot encoded sequences
        mask: (B, fixed_length) True where real bases exist
        Y: (B,) binary labels (0=FALSE, 1=TRUE)
        lengths: (B,) sequence lengths (normalized to [0,1])
    """
    headers, seq_idxs, labels, starts, ends, lengths = zip(*batch)
    
    B = len(batch)
    X = torch.zeros((B, 5, fixed_length), dtype=torch.float32)
    mask = torch.zeros((B, fixed_length), dtype=torch.bool)
    
    for i, (seq_idx, start, end, seq_len) in enumerate(zip(seq_idxs, starts, ends, lengths)):
        actual_len = min(seq_len, fixed_length - start)
        if actual_len > 0:
            idx = torch.from_numpy(seq_idx[:actual_len].astype(np.int64))
            pos = torch.arange(actual_len, dtype=torch.long) + start
            X[i, idx, pos] = 1.0
            mask[i, start:start + actual_len] = (idx != 4)  # Mask out N's
    
    Y = torch.tensor(labels, dtype=torch.long)
    lengths_norm = torch.tensor(lengths, dtype=torch.float32) / fixed_length
    
    return list(headers), X, mask, Y, lengths_norm

## CNN Network for TIR Detection

Key features:
1. **Fixed-length random placement**: Sequences placed at random positions in fixed canvas
2. **Configurable RC fusion**: Early (after first conv) or Late (after pooling)
3. **Multi-scale motif detection**: Different kernel sizes capture TIR features at different scales
4. **Dilated convolutions**: Capture long-range dependencies

In [5]:
# ============ Network Building Blocks ============

class ConvBlock(nn.Module):
    """Residual convolutional block with optional dilation."""
    def __init__(self, c_in, c_out, kernel_size=9, dilation=1, dropout=0.1):
        super().__init__()
        assert kernel_size % 2 == 1
        pad = (kernel_size // 2) * dilation
        self.conv = nn.Conv1d(c_in, c_out, kernel_size, padding=pad, dilation=dilation, bias=True)
        self.bn = nn.BatchNorm1d(c_out)
        self.drop = nn.Dropout(dropout)
        self.proj = nn.Identity() if c_in == c_out else nn.Conv1d(c_in, c_out, 1)

    def forward(self, x):
        y = self.conv(x)
        y = F.gelu(self.bn(y))
        y = self.drop(y)
        return y + self.proj(x)

class MaskedMaxPool1d(nn.Module):
    """Max pooling that respects padding mask."""
    def __init__(self, kernel_size=2, stride=2):
        super().__init__()
        self.kernel_size = kernel_size
        self.stride = stride

    def forward(self, x, mask):
        # Apply mask before pooling
        if mask is not None:
            m = mask.unsqueeze(1).float()
            x = x * m + (~mask.unsqueeze(1)) * (-1e9)  # Large negative for masked positions
        
        x_p = F.max_pool1d(x, self.kernel_size, self.stride)
        
        if mask is None:
            return x_p, None
        m_p = F.max_pool1d(mask.float().unsqueeze(1), self.kernel_size, self.stride).squeeze(1) > 0
        return x_p, m_p

def masked_avg_pool(z, mask):
    """Global average pooling respecting mask."""
    if mask is None:
        return z.mean(-1)
    m = mask.unsqueeze(1).float()
    return (z * m).sum(-1) / m.sum(-1).clamp_min(1.0)

In [6]:
# ============ Early RC Fusion (max after first conv) ============

class RCFirstConv1d(nn.Module):
    """
    RC-invariant first convolution layer.
    Computes max(conv(x), conv(RC(x))) for early RC fusion.
    """
    def __init__(self, out_channels, kernel_size=15, dilation=1, bias=True, dropout=0.1):
        super().__init__()
        assert kernel_size % 2 == 1
        pad = (kernel_size // 2) * dilation
        self.conv = nn.Conv1d(5, out_channels, kernel_size, padding=pad, dilation=dilation, bias=bias)
        self.batch_norm = nn.BatchNorm1d(out_channels)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        y1 = self.conv(x)
        
        # Reverse complement
        x_rc = x.flip(-1).index_select(1, REV_COMP.to(x.device))
        y2 = self.conv(x_rc).flip(-1)
        
        # Take element-wise max for RC invariance
        y = torch.max(y1, y2)
        y = self.batch_norm(y)
        y = F.gelu(y)
        y = self.dropout(y)
        
        return y

## TIR Binary Classification Model

In [7]:
# ============ TIR Binary Classification Model ============

class TIRCNN(nn.Module):
    """
    RC-invariant CNN for TIR presence/absence binary classification.
    
    Architecture:
    - Multi-scale motif detection (different kernel sizes)
    - Dilated convolutions for context
    - RC-invariance via early or late fusion
    - Binary output: TRUE (has TIR) vs FALSE (no TIR)
    """
    def __init__(
        self,
        width=128,
        motif_kernels=(7, 15, 21),
        context_kernel=9,
        context_dilations=(1, 2, 4, 8),
        dropout=0.15,
        rc_mode="early"
    ):
        super().__init__()
        self.rc_mode = rc_mode
        
        # ---- Motif detection layers ----
        if rc_mode == "early":
            self.motif_convs = nn.ModuleList([
                RCFirstConv1d(width, kernel_size=k, dropout=dropout)
                for k in motif_kernels
            ])
        else:
            self.motif_convs = nn.ModuleList([
                nn.Sequential(
                    nn.Conv1d(5, width, kernel_size=k, padding=k//2, bias=True),
                    nn.BatchNorm1d(width),
                    nn.GELU(),
                    nn.Dropout(dropout)
                )
                for k in motif_kernels
            ])
        
        # ---- Mix layer ----
        in_ch = width * len(motif_kernels)
        self.mix = nn.Sequential(
            nn.Conv1d(in_ch, width, kernel_size=1, bias=True),
            nn.BatchNorm1d(width),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        
        # ---- Context blocks ----
        self.context_blocks = nn.ModuleList([
            ConvBlock(width, width, kernel_size=context_kernel, dilation=d, dropout=dropout)
            for d in context_dilations
        ])
        self.pool = MaskedMaxPool1d(kernel_size=2, stride=2)
        
        # ---- Binary classification head ----
        self.classifier = nn.Sequential(
            nn.Linear(width, 128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, 2),  # Binary: 0=FALSE (no TIR), 1=TRUE (has TIR)
        )
        
    @staticmethod
    def rc_transform(x, mask):
        x_rc = x.index_select(1, REV_COMP.to(x.device)).flip(-1)
        mask_rc = None if mask is None else mask.flip(-1)
        return x_rc, mask_rc
    
    def encode(self, x, mask):
        feats = [conv(x) for conv in self.motif_convs]
        z = torch.cat(feats, dim=1)
        z = self.mix(z)
        
        m = mask
        for block in self.context_blocks:
            z = block(z)
            z, m = self.pool(z, m)
        
        return masked_avg_pool(z, m)
    
    def forward(self, x, mask):
        """
        Returns:
            logits: (B, 2) - binary classification logits
        """
        if self.rc_mode == "late":
            f = self.encode(x, mask)
            x_rc, mask_rc = self.rc_transform(x, mask)
            r = self.encode(x_rc, mask_rc)
            pooled = 0.5 * (f + r)
        else:
            pooled = self.encode(x, mask)
        
        logits = self.classifier(pooled)
        return logits

In [8]:
# ============ TIR Training Function ============

def run_train_tir(
    fasta_path,
    label_path,
    batch_size=32,
    epochs=30,
    lr=1e-3,
    width=128,
    motif_kernels=(7, 15, 21),
    context_kernel=9,
    context_dilations=(1, 2, 4, 8),
    dropout=0.15,
    rc_mode=RC_FUSION_MODE,
    label_smoothing=LABEL_SMOOTHING,
    device=None,
    patience=10,
    test_size=0.2,
    random_state=42,
    save_dir=".",
    subset_size=None,       # NEW: Limit total samples (e.g., 30000)
    num_workers=4,          # NEW: DataLoader workers for parallel loading
    prefetch_factor=2,      # NEW: Prefetch batches per worker
):
    """
    Train TIR presence/absence binary classifier.
    
    Task: Predict TRUE (has TIR) vs FALSE (no TIR)
    
    Args:
        subset_size: If set, randomly sample this many sequences (stratified)
        num_workers: Number of DataLoader workers for parallel batch loading
        prefetch_factor: Number of batches to prefetch per worker
    """
    device = resolve_device(device)
    print(f"Using device: {device}")
    print(f"RC fusion mode: {rc_mode}")
    print(f"Canvas size: {FIXED_LENGTH} bp")
    
    # ---- Load data ----
    print("\n=== Loading data ===")
    headers, sequences = read_fasta(fasta_path)
    label_dict = load_tir_labels(label_path)
    
    # Match headers to labels
    all_h, all_s, all_labels = [], [], []
    skipped = 0
    for h, s in zip(headers, sequences):
        if h not in label_dict:
            skipped += 1
            continue
        all_h.append(h)
        all_s.append(s)
        all_labels.append(label_dict[h])
    
    # Free original FASTA data
    del headers, sequences
    gc.collect()
    
    print(f"\nMatched {len(all_h)} sequences (skipped {skipped} without labels)")
    
    all_labels = np.array(all_labels, dtype=np.int64)
    n_true = (all_labels == 1).sum()
    n_false = (all_labels == 0).sum()
    print(f"Full distribution: {n_true} TRUE ({100*n_true/len(all_labels):.1f}%) | {n_false} FALSE ({100*n_false/len(all_labels):.1f}%)")
    
    # ---- Subset sampling (stratified) ----
    if subset_size is not None and subset_size < len(all_h):
        print(f"\n=== Subsampling to {subset_size} samples (stratified) ===")
        
        # Stratified subsample to maintain class balance
        idx_keep, _ = train_test_split(
            np.arange(len(all_h)), 
            train_size=subset_size, 
            stratify=all_labels, 
            random_state=random_state
        )
        
        all_h = [all_h[i] for i in idx_keep]
        all_s = [all_s[i] for i in idx_keep]
        all_labels = all_labels[idx_keep]
        
        n_true = (all_labels == 1).sum()
        n_false = (all_labels == 0).sum()
        print(f"Subset distribution: {n_true} TRUE ({100*n_true/len(all_labels):.1f}%) | {n_false} FALSE ({100*n_false/len(all_labels):.1f}%)")
        
        gc.collect()
    
    # ---- Train/test split ----
    idx_train, idx_test = train_test_split(
        np.arange(len(all_h)), test_size=test_size, stratify=all_labels, random_state=random_state
    )
    
    train_h = [all_h[i] for i in idx_train]
    train_s = [all_s[i] for i in idx_train]
    train_labels = all_labels[idx_train]
    
    test_h = [all_h[i] for i in idx_test]
    test_s = [all_s[i] for i in idx_test]
    test_labels = all_labels[idx_test]
    
    print(f"\nTrain: {len(train_h)}, Test: {len(test_h)}")
    
    # ---- Model ----
    model = TIRCNN(
        width=width,
        motif_kernels=motif_kernels,
        context_kernel=context_kernel,
        context_dilations=context_dilations,
        dropout=dropout,
        rc_mode=rc_mode
    ).to(device)
    
    n_params = sum(p.numel() for p in model.parameters())
    print(f"\nModel parameters: {n_params:,}")
    
    # ---- Loss function with class weights ----
    class_weights = compute_binary_class_weights(train_labels, mode="balanced")
    print(f"Class weights: FALSE={class_weights[0]:.3f}, TRUE={class_weights[1]:.3f}")
    class_weights_t = torch.tensor(class_weights, dtype=torch.float32, device=device)
    
    if label_smoothing > 0:
        loss_fn = nn.CrossEntropyLoss(weight=class_weights_t, label_smoothing=label_smoothing)
    else:
        loss_fn = nn.CrossEntropyLoss(weight=class_weights_t)
    
    # ---- Optimizer ----
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(opt, T_max=epochs, eta_min=lr * 0.01)
    
    # ---- Datasets ----
    print("\n=== Creating datasets ===")
    ds_train = TIRDataset(train_h, train_s, train_labels)
    ds_test = TIRDataset(test_h, test_s, test_labels)
    
    # Free memory
    del train_h, train_s, test_h, test_s
    del all_h, all_s, all_labels
    gc.collect()
    
    # DataLoader with multiple workers for parallel batch loading
    # Use persistent_workers to avoid re-spawning workers each epoch
    use_workers = num_workers if num_workers > 0 else 0
    persistent = use_workers > 0
    
    loader_train = DataLoader(
        ds_train, 
        batch_size=batch_size, 
        shuffle=True, 
        num_workers=use_workers, 
        collate_fn=collate_tir,
        prefetch_factor=prefetch_factor if use_workers > 0 else None,
        persistent_workers=persistent,
        pin_memory=(device.type == "cuda"),
    )
    loader_test = DataLoader(
        ds_test, 
        batch_size=batch_size, 
        shuffle=False, 
        num_workers=use_workers, 
        collate_fn=collate_tir,
        prefetch_factor=prefetch_factor if use_workers > 0 else None,
        persistent_workers=persistent,
        pin_memory=(device.type == "cuda"),
    )
    
    print(f"DataLoader: {use_workers} workers, prefetch={prefetch_factor}, pin_memory={device.type == 'cuda'}")
    
    # ---- Training loop ----
    print("\n=== Training TIR classifier ===")
    history = {
        "train_loss": [],
        "val_acc": [], "val_bal_acc": [], "val_f1": [],
        "val_auroc": [], "val_auprc": []
    }
    best_state, best_epoch = None, None
    best_score = -math.inf
    bad = 0
    
    for ep in range(1, epochs + 1):
        # ---- Train ----
        model.train()
        running_loss = 0.0
        n_samples = 0
        
        pbar = tqdm(loader_train, desc=f"Epoch {ep}/{epochs}", leave=False)
        for _, X, mask, Y, lengths in pbar:
            X = X.to(device, non_blocking=True)
            mask = mask.to(device, non_blocking=True)
            Y = Y.to(device, non_blocking=True)
            
            logits = model(X, mask)
            loss = loss_fn(logits, Y)
            
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            
            running_loss += loss.item() * X.size(0)
            n_samples += X.size(0)
            pbar.set_postfix(loss=f"{loss.item():.4f}")
        
        scheduler.step()
        train_loss = running_loss / n_samples
        
        # ---- Evaluate ----
        model.eval()
        all_pred, all_true, all_probs = [], [], []
        
        with torch.no_grad():
            for _, X, mask, Y, lengths in loader_test:
                X = X.to(device, non_blocking=True)
                mask = mask.to(device, non_blocking=True)
                
                logits = model(X, mask)
                probs = F.softmax(logits, dim=1)[:, 1].cpu().numpy()  # P(TRUE)
                pred = logits.argmax(dim=1).cpu().numpy()
                
                all_pred.extend(pred)
                all_true.extend(Y.numpy())
                all_probs.extend(probs)
        
        all_pred = np.array(all_pred)
        all_true = np.array(all_true)
        all_probs = np.array(all_probs)
        
        acc = accuracy_score(all_true, all_pred)
        bal_acc = balanced_accuracy_score(all_true, all_pred)
        f1 = f1_score(all_true, all_pred, average="binary")
        auroc = roc_auc_score(all_true, all_probs)
        auprc = average_precision_score(all_true, all_probs)
        
        history["train_loss"].append(train_loss)
        history["val_acc"].append(acc)
        history["val_bal_acc"].append(bal_acc)
        history["val_f1"].append(f1)
        history["val_auroc"].append(auroc)
        history["val_auprc"].append(auprc)
        
        print(f"Ep {ep:2d}: loss {train_loss:.4f} | acc {acc:.4f} bal_acc {bal_acc:.4f} F1 {f1:.4f} | AUROC {auroc:.4f} AUPRC {auprc:.4f}")
        
        # Use F1 as primary metric
        if f1 > best_score + 1e-4:
            best_score = f1
            best_epoch = ep
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                print("Early stopping.")
                break
    
    # ---- Final evaluation ----
    print(f"\n{'='*60}")
    print("FINAL EVALUATION")
    print(f"{'='*60}")
    print(f"Best epoch: {best_epoch}")
    
    if best_state is not None:
        model.load_state_dict(best_state)
        model.to(device)
    
    model.eval()
    all_pred, all_true, all_probs = [], [], []
    
    with torch.no_grad():
        for _, X, mask, Y, lengths in loader_test:
            X = X.to(device, non_blocking=True)
            mask = mask.to(device, non_blocking=True)
            
            logits = model(X, mask)
            probs = F.softmax(logits, dim=1)[:, 1].cpu().numpy()
            pred = logits.argmax(dim=1).cpu().numpy()
            
            all_pred.extend(pred)
            all_true.extend(Y.numpy())
            all_probs.extend(probs)
    
    all_pred = np.array(all_pred)
    all_true = np.array(all_true)
    all_probs = np.array(all_probs)
    
    test_acc = accuracy_score(all_true, all_pred)
    test_bal_acc = balanced_accuracy_score(all_true, all_pred)
    test_f1 = f1_score(all_true, all_pred, average="binary")
    test_auroc = roc_auc_score(all_true, all_probs)
    test_auprc = average_precision_score(all_true, all_probs)
    
    print("\n--- Classification Report ---")
    print(classification_report(all_true, all_pred, target_names=["FALSE (no TIR)", "TRUE (has TIR)"], zero_division=0))
    
    print(f"\nAUROC: {test_auroc:.4f}")
    print(f"AUPRC: {test_auprc:.4f}")
    
    # Save checkpoint
    if best_state is not None:
        ckpt = {
            "model_state_dict": best_state,
            "arch": {
                "width": width,
                "motif_kernels": motif_kernels,
                "context_dilations": context_dilations,
                "rc_mode": rc_mode,
            },
            "history": history,
            "best_epoch": best_epoch,
        }
        save_path = Path(save_dir) / "tir_cnn_best.pt"
        torch.save(ckpt, save_path)
        print(f"\nSaved checkpoint to {save_path}")
    
    return {
        "model": model,
        "history": history,
        "best_epoch": best_epoch,
        "best_val_acc": history["val_acc"][best_epoch - 1] if best_epoch else None,
        "best_val_bal_acc": history["val_bal_acc"][best_epoch - 1] if best_epoch else None,
        "best_val_f1": history["val_f1"][best_epoch - 1] if best_epoch else None,
        "best_val_auroc": history["val_auroc"][best_epoch - 1] if best_epoch else None,
        "best_val_auprc": history["val_auprc"][best_epoch - 1] if best_epoch else None,
        "test_acc": test_acc,
        "test_bal_acc": test_bal_acc,
        "test_f1": test_f1,
        "test_auroc": test_auroc,
        "test_auprc": test_auprc,
        "test_pred": all_pred,
        "test_true": all_true,
        "test_probs": all_probs,
        "device": str(device),
    }

In [9]:
# ============ Train TIR Binary Classifier ============

print(f"FASTA: {Path(FASTA_PATH).resolve()}")
print(f"Labels: {Path(LABEL_PATH).resolve()}")

FASTA: /Users/alexyang/Documents/Part III System Biology/TE Classification/data/vgp/all_vgp_tes.fa
Labels: /Users/alexyang/Documents/Part III System Biology/TE Classification/data/vgp/features


## Train Model

## Comparison: Class Weights vs Balanced Sampling

### Approach Analysis

| Method | Data Used | Training Speed | Gradient Stability | Best For |
|--------|-----------|----------------|-------------------|----------|
| **Class Weights** | All 135k | Slower | Can be unstable | When minority class is tiny (<10%) |
| **Balanced Sampling** | ~80k (40k+40k) | ~2x faster | Very stable | Moderate imbalance (30/70) |

**Recommendation for 70/30 split**: Balanced sampling is preferred because:
1. 30% minority = ~40k samples is substantial (no severe data scarcity)
2. Simpler optimization - BCE loss works as intended
3. Faster training (~2x fewer samples)
4. More stable gradients - no need to tune weight strength

**When to use class weights instead**:
- Minority class < 5-10% of data
- Cannot afford to lose majority class examples
- Using techniques like focal loss that already handle imbalance

In [11]:
# ============ Balanced Sampling Strategy ============
# Instead of class weights, undersample majority class to match minority class size

def create_balanced_subset(headers, sequences, labels, random_state=42):
    """
    Create a balanced dataset by undersampling the majority class.
    
    Args:
        headers: list of sequence headers
        sequences: list of sequences
        labels: numpy array of binary labels (0 or 1)
        random_state: random seed for reproducibility
        
    Returns:
        Balanced (headers, sequences, labels)
    """
    np.random.seed(random_state)
    labels = np.asarray(labels)
    
    idx_false = np.where(labels == 0)[0]  # Majority class (FALSE/no TIR)
    idx_true = np.where(labels == 1)[0]   # Minority class (TRUE/has TIR)
    
    n_minority = len(idx_true)
    n_majority = len(idx_false)
    
    print(f"Original distribution:")
    print(f"  FALSE (no TIR):  {n_majority} ({100*n_majority/(n_majority+n_minority):.1f}%)")
    print(f"  TRUE (has TIR):  {n_minority} ({100*n_minority/(n_majority+n_minority):.1f}%)")
    
    # Undersample majority class to match minority class
    idx_false_sampled = np.random.choice(idx_false, size=n_minority, replace=False)
    
    # Combine and shuffle
    idx_balanced = np.concatenate([idx_false_sampled, idx_true])
    np.random.shuffle(idx_balanced)
    
    balanced_h = [headers[i] for i in idx_balanced]
    balanced_s = [sequences[i] for i in idx_balanced]
    balanced_labels = labels[idx_balanced]
    
    print(f"\nBalanced distribution:")
    print(f"  FALSE (no TIR):  {n_minority} (50.0%)")
    print(f"  TRUE (has TIR):  {n_minority} (50.0%)")
    print(f"  Total samples:   {2 * n_minority}")
    print(f"  Data reduction:  {100*(1 - 2*n_minority/(n_majority+n_minority)):.1f}%")
    
    return balanced_h, balanced_s, balanced_labels


# ============ Updated Training Function with Balanced Sampling Option ============

def run_train_tir_v2(
    fasta_path,
    label_path,
    batch_size=32,
    epochs=30,
    lr=1e-3,
    width=128,
    motif_kernels=(7, 15, 21),
    context_kernel=9,
    context_dilations=(1, 2, 4, 8),
    dropout=0.15,
    rc_mode=RC_FUSION_MODE,
    label_smoothing=0.0,  # Default OFF for balanced data
    device=None,
    patience=10,
    test_size=0.2,
    random_state=42,
    save_dir=".",
    balance_mode="undersample",  # "undersample", "weights", or "none"
    class_weight_mode="balanced",  # Only used if balance_mode="weights"
    num_workers=0,
    prefetch_factor=None,
):
    """
    Train TIR presence/absence binary classifier with flexible balancing.
    
    Args:
        balance_mode: 
            - "undersample": Undersample majority class to match minority (recommended)
            - "weights": Use class weights in loss function
            - "none": No balancing (baseline)
        class_weight_mode: Weight calculation mode if balance_mode="weights"
    """
    device = resolve_device(device)
    print(f"Using device: {device}")
    print(f"RC fusion mode: {rc_mode}")
    print(f"Balance mode: {balance_mode}")
    
    # ---- Load data ----
    print("\n=== Loading data ===")
    headers, sequences = read_fasta(fasta_path)
    label_dict = load_tir_labels(label_path)
    
    # Match headers to labels
    all_h, all_s, all_labels = [], [], []
    skipped = 0
    for h, s in zip(headers, sequences):
        if h not in label_dict:
            skipped += 1
            continue
        all_h.append(h)
        all_s.append(s)
        all_labels.append(label_dict[h])
    
    del headers, sequences
    gc.collect()
    
    print(f"\nMatched {len(all_h)} sequences (skipped {skipped} without labels)")
    all_labels = np.array(all_labels, dtype=np.int64)
    
    # ---- Apply balancing strategy ----
    if balance_mode == "undersample":
        print("\n=== Applying balanced undersampling ===")
        all_h, all_s, all_labels = create_balanced_subset(
            all_h, all_s, all_labels, random_state=random_state
        )
        use_class_weights = False
    elif balance_mode == "weights":
        print("\n=== Using class weights (no undersampling) ===")
        use_class_weights = True
    else:
        print("\n=== No balancing applied ===")
        use_class_weights = False
    
    # ---- Train/test split (stratified) ----
    idx_train, idx_test = train_test_split(
        np.arange(len(all_h)), test_size=test_size, stratify=all_labels, random_state=random_state
    )
    
    train_h = [all_h[i] for i in idx_train]
    train_s = [all_s[i] for i in idx_train]
    train_labels = all_labels[idx_train]
    
    test_h = [all_h[i] for i in idx_test]
    test_s = [all_s[i] for i in idx_test]
    test_labels = all_labels[idx_test]
    
    print(f"\nTrain: {len(train_h)}, Test: {len(test_h)}")
    print(f"Train distribution: {(train_labels==1).sum()} TRUE / {(train_labels==0).sum()} FALSE")
    print(f"Test distribution:  {(test_labels==1).sum()} TRUE / {(test_labels==0).sum()} FALSE")
    
    # ---- Model ----
    model = TIRCNN(
        width=width,
        motif_kernels=motif_kernels,
        context_kernel=context_kernel,
        context_dilations=context_dilations,
        dropout=dropout,
        rc_mode=rc_mode
    ).to(device)
    
    n_params = sum(p.numel() for p in model.parameters())
    print(f"\nModel parameters: {n_params:,}")
    
    # ---- Loss function ----
    if use_class_weights:
        class_weights = compute_binary_class_weights(train_labels, mode=class_weight_mode)
        print(f"Class weights ({class_weight_mode}): FALSE={class_weights[0]:.3f}, TRUE={class_weights[1]:.3f}")
        class_weights_t = torch.tensor(class_weights, dtype=torch.float32, device=device)
        loss_fn = nn.CrossEntropyLoss(weight=class_weights_t, label_smoothing=label_smoothing)
    else:
        print("Using unweighted CrossEntropyLoss (balanced data)")
        if label_smoothing > 0:
            loss_fn = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
        else:
            loss_fn = nn.CrossEntropyLoss()
    
    # ---- Optimizer ----
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(opt, T_max=epochs, eta_min=lr * 0.01)
    
    # ---- Datasets ----
    ds_train = TIRDataset(train_h, train_s, train_labels)
    ds_test = TIRDataset(test_h, test_s, test_labels)
    
    del train_h, train_s, test_h, test_s, all_h, all_s, all_labels
    gc.collect()
    
    loader_train = DataLoader(
        ds_train, batch_size=batch_size, shuffle=True, 
        num_workers=num_workers, collate_fn=collate_tir,
        pin_memory=(device.type == "cuda"),
    )
    loader_test = DataLoader(
        ds_test, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, collate_fn=collate_tir,
        pin_memory=(device.type == "cuda"),
    )
    
    print(f"\nBatches per epoch: {len(loader_train)}")
    
    # ---- Training loop ----
    print("\n=== Training TIR classifier ===")
    history = {
        "train_loss": [],
        "val_acc": [], "val_bal_acc": [], "val_f1": [],
        "val_auroc": [], "val_auprc": []
    }
    best_state, best_epoch = None, None
    best_score = -math.inf
    bad = 0
    
    for ep in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        n_samples = 0
        
        pbar = tqdm(loader_train, desc=f"Epoch {ep}/{epochs}", leave=False)
        for _, X, mask, Y, lengths in pbar:
            X = X.to(device, non_blocking=True)
            mask = mask.to(device, non_blocking=True)
            Y = Y.to(device, non_blocking=True)
            
            logits = model(X, mask)
            loss = loss_fn(logits, Y)
            
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            
            running_loss += loss.item() * X.size(0)
            n_samples += X.size(0)
            pbar.set_postfix(loss=f"{loss.item():.4f}")
        
        scheduler.step()
        train_loss = running_loss / n_samples
        
        # ---- Evaluate ----
        model.eval()
        all_pred, all_true, all_probs = [], [], []
        
        with torch.no_grad():
            for _, X, mask, Y, lengths in loader_test:
                X = X.to(device, non_blocking=True)
                mask = mask.to(device, non_blocking=True)
                
                logits = model(X, mask)
                probs = F.softmax(logits, dim=1)[:, 1].cpu().numpy()
                pred = logits.argmax(dim=1).cpu().numpy()
                
                all_pred.extend(pred)
                all_true.extend(Y.numpy())
                all_probs.extend(probs)
        
        all_pred = np.array(all_pred)
        all_true = np.array(all_true)
        all_probs = np.array(all_probs)
        
        acc = accuracy_score(all_true, all_pred)
        bal_acc = balanced_accuracy_score(all_true, all_pred)
        f1 = f1_score(all_true, all_pred, average="binary")
        auroc = roc_auc_score(all_true, all_probs)
        auprc = average_precision_score(all_true, all_probs)
        
        history["train_loss"].append(train_loss)
        history["val_acc"].append(acc)
        history["val_bal_acc"].append(bal_acc)
        history["val_f1"].append(f1)
        history["val_auroc"].append(auroc)
        history["val_auprc"].append(auprc)
        
        print(f"Ep {ep:2d}: loss {train_loss:.4f} | acc {acc:.4f} bal_acc {bal_acc:.4f} F1 {f1:.4f} | AUROC {auroc:.4f} AUPRC {auprc:.4f}")
        
        if f1 > best_score + 1e-4:
            best_score = f1
            best_epoch = ep
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                print("Early stopping.")
                break
    
    # ---- Final evaluation ----
    if best_state is not None:
        model.load_state_dict(best_state)
        model.to(device)
    
    model.eval()
    all_pred, all_true, all_probs = [], [], []
    
    with torch.no_grad():
        for _, X, mask, Y, lengths in loader_test:
            X = X.to(device, non_blocking=True)
            mask = mask.to(device, non_blocking=True)
            
            logits = model(X, mask)
            probs = F.softmax(logits, dim=1)[:, 1].cpu().numpy()
            pred = logits.argmax(dim=1).cpu().numpy()
            
            all_pred.extend(pred)
            all_true.extend(Y.numpy())
            all_probs.extend(probs)
    
    all_pred = np.array(all_pred)
    all_true = np.array(all_true)
    all_probs = np.array(all_probs)
    
    return {
        "model": model,
        "history": history,
        "best_epoch": best_epoch,
        "best_state": best_state,
        "test_acc": accuracy_score(all_true, all_pred),
        "test_bal_acc": balanced_accuracy_score(all_true, all_pred),
        "test_f1": f1_score(all_true, all_pred, average="binary"),
        "test_auroc": roc_auc_score(all_true, all_probs),
        "test_auprc": average_precision_score(all_true, all_probs),
        "test_pred": all_pred,
        "test_true": all_true,
        "test_probs": all_probs,
        "balance_mode": balance_mode,
    }

In [12]:
# ============ Run Comparison: Balanced Sampling vs Class Weights ============
# Train both approaches and compare performance

comparison_results = {}

# ---- Approach 1: Balanced Sampling (recommended) ----
print("=" * 70)
print("APPROACH 1: BALANCED UNDERSAMPLING")
print("=" * 70)

results_balanced = run_train_tir_v2(
    fasta_path=FASTA_PATH,
    label_path=LABEL_PATH,
    batch_size=BATCH_SIZE,
    epochs=15,  # Quick comparison
    lr=1e-3,
    width=128,
    motif_kernels=(7, 15, 21),
    context_dilations=(1, 2, 4, 8),
    dropout=0.15,
    rc_mode=RC_FUSION_MODE,
    label_smoothing=0.0,  # No smoothing for balanced data
    device=DEVICE,
    patience=10,
    balance_mode="undersample",  # <-- Undersampling
    random_state=42,
)
comparison_results["balanced_sampling"] = results_balanced

print("\n" + "=" * 70)
print("APPROACH 2: CLASS WEIGHTS (inv_sqrt)")
print("=" * 70)

results_weights = run_train_tir_v2(
    fasta_path=FASTA_PATH,
    label_path=LABEL_PATH,
    batch_size=BATCH_SIZE,
    epochs=15,  # Quick comparison
    lr=3e-4,  # Lower LR for weighted loss stability
    width=128,
    motif_kernels=(7, 15, 21),
    context_dilations=(1, 2, 4, 8),
    dropout=0.15,
    rc_mode=RC_FUSION_MODE,
    label_smoothing=0.0,  # No smoothing
    device=DEVICE,
    patience=10,
    balance_mode="weights",  # <-- Class weights
    class_weight_mode="inv_sqrt",
    random_state=42,
)
comparison_results["class_weights"] = results_weights

APPROACH 1: BALANCED UNDERSAMPLING
Using device: mps
RC fusion mode: early
Balance mode: undersample

=== Loading data ===
Loaded 135751 TIR labels:
  ✓ TRUE (has TIR):  40424 (29.8%)
  ✗ FALSE (no TIR):  95327 (70.2%)

Matched 135751 sequences (skipped 0 without labels)

=== Applying balanced undersampling ===
Original distribution:
  FALSE (no TIR):  95327 (70.2%)
  TRUE (has TIR):  40424 (29.8%)

Balanced distribution:
  FALSE (no TIR):  40424 (50.0%)
  TRUE (has TIR):  40424 (50.0%)
  Total samples:   80848
  Data reduction:  40.4%

Train: 64678, Test: 16170
Train distribution: 32339 TRUE / 32339 FALSE
Test distribution:  8085 TRUE / 8085 FALSE

Model parameters: 686,338
Using unweighted CrossEntropyLoss (balanced data)

Batches per epoch: 2022

=== Training TIR classifier ===


Epoch 1/15:   0%|          | 0/2022 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# ============ Compare Results ============

print("\n" + "=" * 70)
print("COMPARISON SUMMARY")
print("=" * 70)

metrics = ["test_acc", "test_bal_acc", "test_f1", "test_auroc", "test_auprc"]
metric_names = ["Accuracy", "Balanced Acc", "F1 Score", "AUROC", "AUPRC"]

print(f"\n{'Metric':<20} {'Balanced Sampling':<20} {'Class Weights':<20} {'Winner':<15}")
print("-" * 75)

for m, name in zip(metrics, metric_names):
    v_balanced = comparison_results["balanced_sampling"][m]
    v_weights = comparison_results["class_weights"][m]
    diff = v_balanced - v_weights
    
    if abs(diff) < 0.01:
        winner = "≈ Tie"
    elif diff > 0:
        winner = "Balanced ✓"
    else:
        winner = "Weights ✓"
    
    print(f"{name:<20} {v_balanced:.4f}{'':<14} {v_weights:.4f}{'':<14} {winner}")

print("\n" + "-" * 75)
print("\nTraining efficiency:")
n_samples_balanced = len(comparison_results["balanced_sampling"]["test_true"]) * 5  # Approx train size
n_samples_weights = len(comparison_results["class_weights"]["test_true"]) * 5

# Plot comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# F1 curves
ax1 = axes[0]
epochs_b = range(1, len(comparison_results["balanced_sampling"]["history"]["val_f1"]) + 1)
epochs_w = range(1, len(comparison_results["class_weights"]["history"]["val_f1"]) + 1)
ax1.plot(epochs_b, comparison_results["balanced_sampling"]["history"]["val_f1"], 'b-', label='Balanced Sampling', linewidth=2)
ax1.plot(epochs_w, comparison_results["class_weights"]["history"]["val_f1"], 'r-', label='Class Weights', linewidth=2)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("F1 Score")
ax1.set_title("F1 Score Comparison")
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_ylim([0, 1])

# AUROC curves
ax2 = axes[1]
ax2.plot(epochs_b, comparison_results["balanced_sampling"]["history"]["val_auroc"], 'b-', label='Balanced Sampling', linewidth=2)
ax2.plot(epochs_w, comparison_results["class_weights"]["history"]["val_auroc"], 'r-', label='Class Weights', linewidth=2)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("AUROC")
ax2.set_title("AUROC Comparison")
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_ylim([0, 1])

# Loss curves
ax3 = axes[2]
ax3.plot(epochs_b, comparison_results["balanced_sampling"]["history"]["train_loss"], 'b-', label='Balanced Sampling', linewidth=2)
ax3.plot(epochs_w, comparison_results["class_weights"]["history"]["train_loss"], 'r-', label='Class Weights', linewidth=2)
ax3.set_xlabel("Epoch")
ax3.set_ylabel("Training Loss")
ax3.set_title("Training Loss Comparison")
ax3.legend()
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("tir_balancing_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

# Recommendation
print("\n" + "=" * 70)
print("RECOMMENDATION")
print("=" * 70)
f1_balanced = comparison_results["balanced_sampling"]["test_f1"]
f1_weights = comparison_results["class_weights"]["test_f1"]

if f1_balanced > f1_weights + 0.02:
    print("✓ Balanced Sampling performs better - use balance_mode='undersample'")
elif f1_weights > f1_balanced + 0.02:
    print("✓ Class Weights performs better - use balance_mode='weights'")
else:
    print("≈ Both approaches perform similarly")
    print("  Balanced Sampling is preferred for:")
    print("    - Faster training (fewer samples)")
    print("    - Simpler loss dynamics")
    print("    - No hyperparameter tuning for weights")

In [ ]:
# ============ Run Training with Balanced Sampling ============
# Using balanced undersampling (recommended for 70/30 imbalance)

results = run_train_tir_v2(
    fasta_path=FASTA_PATH,
    label_path=LABEL_PATH,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    lr=LR,
    width=128,
    motif_kernels=(7, 15, 21),
    context_dilations=(1, 2, 4, 8),
    dropout=0.15,
    rc_mode=RC_FUSION_MODE,
    label_smoothing=0.0,  # No smoothing needed for balanced data
    device=DEVICE,
    patience=PATIENCE,
    test_size=0.2,
    random_state=42,
    save_dir=".",
    # Balancing strategy
    balance_mode="undersample",  # Recommended: undersample majority class
    # balance_mode="weights",    # Alternative: use class weights
    # class_weight_mode="inv_sqrt",  # Only used if balance_mode="weights"
    num_workers=0,  # Must be 0 for notebooks on macOS
)

Using device: mps
RC fusion mode: early
Canvas size: 40000 bp

=== Loading data ===
Loaded 135751 TIR labels:
  ✓ TRUE (has TIR):  40424 (29.8%)
  ✗ FALSE (no TIR):  95327 (70.2%)

Matched 135751 sequences (skipped 0 without labels)
Full distribution: 40424 TRUE (29.8%) | 95327 FALSE (70.2%)

=== Subsampling to 30000 samples (stratified) ===
Subset distribution: 8933 TRUE (29.8%) | 21067 FALSE (70.2%)

Train: 24000, Test: 6000

Model parameters: 686,338
Class weights: FALSE=0.595, TRUE=1.405

=== Creating datasets ===
DataLoader: 0 workers, prefetch=None, pin_memory=False

=== Training TIR classifier ===


Epoch 1/1:   0%|          | 0/750 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# ============ Plot Training Curves ============

history = results["history"]
epochs_range = range(1, len(history["train_loss"]) + 1)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("TIR Binary Classification Training History", fontsize=14, fontweight="bold")

# Training loss
ax1 = axes[0, 0]
ax1.plot(epochs_range, history["train_loss"], "b-", label="Training Loss", linewidth=2)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Training Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy metrics
ax2 = axes[0, 1]
ax2.plot(epochs_range, history["val_acc"], "b-", label="Accuracy", linewidth=2)
ax2.plot(epochs_range, history["val_bal_acc"], "g-", label="Balanced Accuracy", linewidth=2)
ax2.plot(epochs_range, history["val_f1"], "r-", label="F1 Score", linewidth=2)
ax2.axvline(x=results["best_epoch"], color="gray", linestyle="--", alpha=0.7, label=f"Best Epoch ({results['best_epoch']})")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Score")
ax2.set_title("Classification Metrics")
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_ylim([0, 1])

# ROC and PR curves
ax3 = axes[1, 0]
ax3.plot(epochs_range, history["val_auroc"], "b-", label="AUROC", linewidth=2)
ax3.plot(epochs_range, history["val_auprc"], "r-", label="AUPRC", linewidth=2)
ax3.axvline(x=results["best_epoch"], color="gray", linestyle="--", alpha=0.7)
ax3.set_xlabel("Epoch")
ax3.set_ylabel("Score")
ax3.set_title("AUROC and AUPRC")
ax3.legend()
ax3.grid(True, alpha=0.3)
ax3.set_ylim([0, 1])

# Confusion matrix
ax4 = axes[1, 1]
cm = confusion_matrix(results["test_true"], results["test_pred"])
import seaborn as sns
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax4,
            xticklabels=["FALSE", "TRUE"],
            yticklabels=["FALSE", "TRUE"])
ax4.set_xlabel("Predicted")
ax4.set_ylabel("True")
ax4.set_title("Confusion Matrix")

plt.tight_layout()
plt.savefig("tir_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## Final Results Summary

In [ ]:
# ============ Print Final Summary ============

print("=" * 60)
print("TIR BINARY CLASSIFICATION - FINAL RESULTS")
print("=" * 60)
print()

print(f"Best Epoch: {results['best_epoch']}")
print()

print("VALIDATION METRICS (at best epoch):")
print(f"  Accuracy:          {results['best_val_acc']:.4f}")
print(f"  Balanced Accuracy: {results['best_val_bal_acc']:.4f}")
print(f"  F1 Score:          {results['best_val_f1']:.4f}")
print(f"  AUROC:             {results['best_val_auroc']:.4f}")
print(f"  AUPRC:             {results['best_val_auprc']:.4f}")
print()

print("HELD-OUT TEST SET METRICS:")
print(f"  Accuracy:          {results['test_acc']:.4f}")
print(f"  Balanced Accuracy: {results['test_bal_acc']:.4f}")
print(f"  F1 Score:          {results['test_f1']:.4f}")
print(f"  AUROC:             {results['test_auroc']:.4f}")
print(f"  AUPRC:             {results['test_auprc']:.4f}")
print()

print("CLASSIFICATION REPORT (Test Set):")
print(classification_report(results["test_true"], results["test_pred"], 
                            target_names=["FALSE (No TIR)", "TRUE (Has TIR)"]))

print("=" * 60)

## Error Analysis

In [ ]:
# ============ Analyze Errors ============

# Get test predictions and probabilities
test_true = np.array(results["test_true"])
test_pred = np.array(results["test_pred"])
test_probs = np.array(results["test_probs"])

# Find false positives (predicted TRUE but actually FALSE)
fp_mask = (test_pred == 1) & (test_true == 0)
fp_indices = np.where(fp_mask)[0]

# Find false negatives (predicted FALSE but actually TRUE)  
fn_mask = (test_pred == 0) & (test_true == 1)
fn_indices = np.where(fn_mask)[0]

print(f"False Positives: {len(fp_indices)} (predicted TIR when no TIR)")
print(f"False Negatives: {len(fn_indices)} (missed TIR when TIR present)")
print()

# Show confidence distribution for errors
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

if len(fp_indices) > 0:
    axes[0].hist(test_probs[fp_indices], bins=20, edgecolor='black', alpha=0.7, color='red')
    axes[0].set_xlabel("Predicted Probability (TIR)")
    axes[0].set_ylabel("Count")
    axes[0].set_title(f"False Positives (n={len(fp_indices)})\nModel Confidence Distribution")
    axes[0].axvline(0.5, color='black', linestyle='--', label='Threshold')
    axes[0].legend()
else:
    axes[0].text(0.5, 0.5, "No False Positives!", ha='center', va='center', fontsize=14)
    axes[0].set_title("False Positives")

if len(fn_indices) > 0:
    axes[1].hist(test_probs[fn_indices], bins=20, edgecolor='black', alpha=0.7, color='orange')
    axes[1].set_xlabel("Predicted Probability (TIR)")
    axes[1].set_ylabel("Count")
    axes[1].set_title(f"False Negatives (n={len(fn_indices)})\nModel Confidence Distribution")
    axes[1].axvline(0.5, color='black', linestyle='--', label='Threshold')
    axes[1].legend()
else:
    axes[1].text(0.5, 0.5, "No False Negatives!", ha='center', va='center', fontsize=14)
    axes[1].set_title("False Negatives")

plt.tight_layout()
plt.savefig("tir_error_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

# Print some examples
print("\n--- Sample False Positives (highest confidence) ---")
if len(fp_indices) > 0:
    fp_sorted = fp_indices[np.argsort(test_probs[fp_indices])[::-1][:5]]
    for idx in fp_sorted:
        print(f"  Index {idx}: prob={test_probs[idx]:.4f}")

print("\n--- Sample False Negatives (lowest confidence) ---")
if len(fn_indices) > 0:
    fn_sorted = fn_indices[np.argsort(test_probs[fn_indices])[:5]]
    for idx in fn_sorted:
        print(f"  Index {idx}: prob={test_probs[idx]:.4f}")